# Continuous Binning — Parameter Sensitivity Analysis

Sweeps the bias node parameter grid defined in `config.py`, fits a `ContinuousBinningModel`
across both feature params and `n_bins`, and runs `generate_parameter_sensitivity_report()`
to identify stable regions and produce interactive visualizations.

**Workflow:**
1. Edit `config.py` (or override inline below) to set tickers, date range, bias_spec
2. Run all cells (Kernel -> Restart & Run All)
3. Interact with the Plotly figures in Section 5


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
from datetime import datetime
from dataclasses import replace

from feature_research.continuous_binning.config import load_config
from feature_research.continuous_binning.data_loader import (
    expand_bias_specs,
    load_features_for_combo,
    populate_cache_if_needed,
    param_combo_label,
)
from feature_selection.base_models.continuous_binning import ContinuousBinningModel
from metrics.performance import SortinoRatio
from eda.parameter_analysis import generate_parameter_sensitivity_report
from utils.enums import Ticker, TimeFrame

print("Imports OK")

Imports OK


## 1. Configuration

Edit `config.py` for persistent changes, or use the override cell below for one-off runs.
`dataclasses.replace()` is used because `ResearchConfig` is a frozen dataclass.

In [2]:
# --- Load defaults from config.py ---
config = load_config()

# ============================================================
# OVERRIDE CELL — edit here to customise without touching config.py
# ============================================================

# config = replace(config, tickers=[Ticker.ES])
# config = replace(config, start=datetime(2015, 1, 1), end=datetime(2024, 12, 31))
# config = replace(config,
#     bias_spec={
#         "module_name": "rsi",
#         "timeframes": [TimeFrame.D],
#         "params": {"lookback": [5, 10, 14, 20, 30]},
#     }
# )

# --- Param sensitivity knobs ---
STABILITY_THRESHOLD = 0.8
TOP_K = 3
METRIC_COL = "sortino"
PLOT_3D_MODE = "surface_slices"  # options: "heatmap_slices", "surface_slices"
N_BINS_GRID = list(config.binning_params.bin_counts)

# ============================================================

if not N_BINS_GRID:
    raise ValueError(
        "binning_params.bin_counts is empty. Provide at least one n_bins value "
        "for parameter sensitivity."
    )

# Derive varying param names (list-valued params only)
varying_params = [
    k for k, v in config.bias_spec["params"].items()
    if isinstance(v, list) and len(v) > 1
]
if not varying_params:
    print(
        "WARNING: No list-valued params found in bias_spec. "
        "Parameter sensitivity requires at least one param with multiple values. "
        "Results will be a single-point grid for feature params; only n_bins will vary."
    )
    varying_params = list(config.bias_spec["params"].keys())

report_param_names = [*varying_params, "n_bins"]

print(f"Module      : {config.bias_spec['module_name']}")
print(f"Tickers     : {[t.name for t in config.tickers]}")
print(f"Period      : {config.start.date()} -> {config.end.date()}")
print(f"Varying     : {varying_params}")
print(f"n_bins grid : {N_BINS_GRID}")
print(f"METRIC      : {METRIC_COL}  |  THRESHOLD: {STABILITY_THRESHOLD}")
print(f"3D MODE     : {PLOT_3D_MODE}")



Module      : rsi
Tickers     : ['ES', 'NQ', 'YM', 'RTY']
Period      : 2000-01-01 -> 2024-12-31
Varying     : ['lookback']
n_bins grid : [10, 9, 8, 7, 6, 5, 4, 3]
METRIC      : sortino  |  THRESHOLD: 0.8


## 2. Feature Extraction

Loads feature + target data for every param combo. Uses the cache if `config.use_cache=True`.
Set `config.populate_cache=True` to auto-populate missing cache entries.

In [3]:
populate_cache_if_needed(config)

expanded_specs = expand_bias_specs(config.bias_spec)
print(f"Parameter grid: {len(expanded_specs)} combinations\n")

combo_data = {}
for spec in expanded_specs:
    result = load_features_for_combo(spec, config)
    if result is not None:
        feature, target, feature_col = result
        combo_data[param_combo_label(spec["params"])] = {
            "feature": feature,
            "target": target,
            "params": spec["params"],
        }
    else:
        print(f"  SKIP: {spec['params']}")

print(f"\nLoaded {len(combo_data)}/{len(expanded_specs)} combos successfully")

Partial cache coverage for rsi (ES, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for rsi (NQ, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for rsi (YM, D). Cache starts at 2002-04-05 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for rsi (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for atr (ES, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for atr (NQ, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for atr (YM, D). Cache starts at 2002-04-05 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.


  Adding required auxiliary: atr (for log_return_atr/log_return_ewsd)
  Adding required auxiliary: ewsd (for log_return_atr/log_return_ewsd)
Populating cache for 44 combinations...
  Bias node specs: 11 (including required atr/ewsd if needed)
  Tickers: 4
  Date range: 2000-01-01 00:00:00 to 2024-12-31 00:00:00
  Max workers: 4
  [1/44] rsi / RTY / D
  [2/44] rsi / YM / D
  [3/44] rsi / NQ / D
  [4/44] rsi / ES / D
  [5/44] rsi / NQ / D
  [6/44] rsi / RTY / D
  [7/44] rsi / ES / D
  [8/44] rsi / YM / D
  [9/44] rsi / RTY / D
  [10/44] rsi / NQ / D
  [11/44] rsi / ES / D
  [12/44] rsi / YM / D
  [13/44] rsi / RTY / D
  [14/44] rsi / YM / D
  [15/44] rsi / NQ / D
  [16/44] rsi / ES / D
  [17/44] rsi / ES / D
  [18/44] rsi / YM / D
  [19/44] rsi / NQ / D
  [20/44] rsi / RTY / D
  [21/44] rsi / ES / D
  [22/44] rsi / RTY / D
  [23/44] rsi / NQ / D
  [24/44] rsi / YM / D
  [25/44] rsi / YM / D
  [26/44] rsi / NQ / D
  [27/44] rsi / ES / D
  [28/44] rsi / RTY / D
  [29/44] rsi / NQ / D
  [30

Partial cache coverage for atr (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for  (ES, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for  (NQ, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for  (YM, D). Cache starts at 2002-04-05 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for  (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for rsi (ES, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for rsi (ES, D). Cache ends at 2023-12-29 00:00:00, requested 2024-12-31 00:00:00. Proceeding with available data.
Partial cache coverage for rsi (


Loaded 9/9 combos successfully


## 3. Metric Grid

Fits `ContinuousBinningModel` for each `(feature params, n_bins)` combo and computes the
Sortino ratio on selected returns. Builds `results_df` with `param{K}_value` + `n_bins`
columns required by `generate_parameter_sensitivity_report()`.


In [4]:
metric = SortinoRatio(annualization_factor=252)
rows = []

for spec in expanded_specs:
    label = param_combo_label(spec["params"])
    if label not in combo_data:
        continue

    data = combo_data[label]
    feature = data["feature"]
    target = data["target"]

    for n_bins in N_BINS_GRID:
        try:
            model = ContinuousBinningModel(n_bins=n_bins)
            model.fit(feature, target)
            signals = model.predict(feature, strategy="long")
            signals = np.asarray(signals).astype(bool).flatten()
            selected_returns = target.values[signals]

            if len(selected_returns) < 5:
                print(
                    f"  SKIP [{label}, n_bins={n_bins}]: too few signals "
                    f"({len(selected_returns)}) to compute reliable metric"
                )
                continue

            metric_value = metric.compute(selected_returns)

            # Columns must be param1_value, param2_value, ... for report (param_names → paramK_value)
            row = {}
            for k, name in enumerate(report_param_names):
                if name == "n_bins":
                    row[f"param{k+1}_value"] = n_bins
                else:
                    row[f"param{k+1}_value"] = spec["params"][name]
            row[METRIC_COL] = metric_value
            rows.append(row)

        except Exception as e:
            print(f"  ERROR [{label}, n_bins={n_bins}]: {e}")

results_df = pd.DataFrame(rows)
print(f"Grid results: {len(results_df)} rows")
results_df


Grid results: 72 rows


,param1_value,param2_value,sortino
0,2,10,1.771089
1,2,9,1.644561
2,2,8,1.673616
3,2,7,1.524472
4,2,6,1.467493
...,...,...,...
67,10,7,1.181490
68,10,6,0.842005
69,10,5,0.652102
70,10,4,0.479832


## 4. Parameter Sensitivity Report

Runs: neighbor smoothing → stable region detection → recommendations.

In [5]:
if results_df.empty:
    raise RuntimeError(
        f"No results collected. Check that combos loaded: "
        f"{len(combo_data)}/{len(expanded_specs)} loaded. "
        f"Ensure cache is available or set config.populate_cache=True."
    )

report = generate_parameter_sensitivity_report(
    results_df=results_df,
    param_names=report_param_names,
    metric_col=METRIC_COL,
    stability_threshold=STABILITY_THRESHOLD,
    top_k=TOP_K,
    plot_3d_mode=PLOT_3D_MODE,
)

print(f"Module              : {config.bias_spec['module_name']}")
print(f"Metric              : {report.metric_name}")
print(f"Grid size           : {report.n_parameter_combinations}")
print(f"Stable regions      : {report.n_stable_regions}")
print(f"Mean stability ratio: {report.mean_stability_ratio:.3f}")
print(f"% stable combos     : {report.pct_stable_combinations:.1%}")
print(f"\nTop-{TOP_K} recommendations:")
for i, combo in enumerate(report.top_k_combinations, 1):
    params_str = ", ".join(
        f"{name}={val}" for name, val in zip(report_param_names, combo)
    )
    print(f"  {i}. {params_str}")




Module              : rsi
Metric              : sortino
Grid size           : 72
Stable regions      : 1
Mean stability ratio: 1.014
% stable combos     : 100.0%

Top-3 recommendations:
  1. lookback=5, n_bins=10
  2. lookback=4, n_bins=10
  3. lookback=6, n_bins=10


## 5. Interactive Visualizations

The appropriate plot is selected automatically based on grid dimensionality:
- 1D grid → raw + smoothed line with shaded stable regions
- 2D grid → heatmap with stability contour overlay
- 3D+ grid → heatmap slices with dropdown + slider controls

In [6]:
if report.plot_1d is not None:
    report.plot_1d.show()

if report.plot_2d is not None:
    report.plot_2d.show()

if report.plot_3d is not None:
    report.plot_3d.show()

## 6. Optional Save

Uncomment cells below to save plots as interactive HTML or grid results as parquet.

In [ ]:
# --- OPTIONAL: Save outputs (uncomment to enable) ---

# from pathlib import Path
# save_dir = config.reports_dir / "param_sensitivity"
# save_dir.mkdir(parents=True, exist_ok=True)

# if report.plot_1d is not None:
#     report.plot_1d.write_html(save_dir / f"param_sensitivity_1d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 1D plot -> {save_dir}")

# if report.plot_2d is not None:
#     report.plot_2d.write_html(save_dir / f"param_sensitivity_2d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 2D plot -> {save_dir}")

# if report.plot_3d is not None:
#     report.plot_3d.write_html(save_dir / f"param_sensitivity_3d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 3D plot -> {save_dir}")

# report.grid_results.to_parquet(save_dir / "param_sensitivity_grid.parquet")
# print(f"Saved grid results -> {save_dir / 'param_sensitivity_grid.parquet'}")